In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats

In [62]:
df_fixtures = pd.read_csv("wc_2026_fixtures_clean.csv")
df_teams = pd.read_csv("wc_2026_teams_clean.csv")
df_editions = pd.read_csv("wc_all_editions_clean.csv")
df_matches = pd.read_csv("wc_all_matches_clean.csv")
df_scorers = pd.read_csv("wc_top_scorers_clean.csv")
dfs = {"fixtures":df_fixtures, "teams":df_teams, "editions":df_editions, "matches":df_matches, "scorers":df_scorers}

In [63]:
df_fixtures["venue"].unique()

<StringArray>
[         'Estadio Azteca',           'Estadio Akron',
   'Mercedes-Benz Stadium',            'Estadio BBVA',
               'BMO Field',          'Levi's Stadium',
            'SoFi Stadium',                'BC Place',
         'MetLife Stadium',        'Gillette Stadium',
 'Lincoln Financial Field',       'Hard Rock Stadium',
             'NRG Stadium',       'Arrowhead Stadium',
            'AT&T Stadium',             'Lumen Field']
Length: 16, dtype: str

In [64]:
for name, df in dfs.items():
    print(f"Columns in {name}: {df.columns.to_list()}\n")

Columns in fixtures: ['group', 'stage', 'team1', 'team2', 'venue', 'city', 'country', 'date', 'kickoff_et', 'team1_confederation', 'team1_fifa_rank', 'team1_coach', 'team2_confederation', 'team2_fifa_rank', 'team2_coach', 'manually_corrected']

Columns in teams: ['team', 'group', 'confederation', 'fifa_rank', 'coach', 'best_wc_result', 'debut_2026', 'confederation_full']

Columns in editions: ['edition', 'year', 'host', 'champion', 'runner_up', 'third_place', 'fourth_place', 'teams', 'matches', 'goals', 'goals_per_match', 'attendance', 'top_scorer', 'top_scorer_country', 'top_scorer_goals', 'start_date', 'end_date', 'final_city', 'host_won', 'format', 'format_raw', 'host_won_raw', 'champion_canon', 'runner_up_canon', 'third_place_canon', 'fourth_place_canon', 'top_scorer_country_canon', 'host_canon']

Columns in matches: ['year', 'stage', 'team1', 'score1', 'score2', 'team2', 'venue', 'city', 'country', 'date', 'notes', 'manually_corrected', 'team1_canon', 'team2_canon', 'country_canon

## Data Analysis

### part 1

In [5]:
print(df_editions["edition"].count())

22


In [6]:
df_fixtures.shape[0]

104

In [7]:
total_goals = df_matches["total_goals"].sum() # not accurate since only 184 matches are included in the dataset, but there have been 900 matches in total.
print(total_goals)

647


In [8]:
avg_goals = total_goals / df_matches.shape[0]
print(f"{avg_goals:.2f}")

3.52


In [9]:
df_editions["attendance"].sum() / df_editions["attendance"].count()

np.float64(1981512.9090909092)

In [10]:
df_editions["host"].nunique()

18

In [11]:
teams = set(df_matches["team1"]) | set(df_matches["team2"])
print(len(teams)) # not accurate since only 184 matches are included in the dataset, but there have been 900 matches in total.

54


### part 2

In [12]:
goals_per_year = df_editions[["year","goals", "edition"]]
fig = px.line(
    goals_per_year, 
    x="year",
    y="goals",
    title="Total Goals per Year",
    markers=True
)

fig.show()

In [13]:
attendance_per_year = df_editions[["year","attendance", "edition"]]

fig = px.line(
    attendance_per_year, 
    x="year",
    y="attendance",
    title="Total Attendance per Year",
    markers=True
)

fig.show()

In [14]:
goals_per_match_per_year = df_editions[["year","goals_per_match"]]
fig = px.line(
    goals_per_match_per_year, 
    x="year",
    y="goals_per_match",
    title="Total (Goals per match) per Year",
    markers=True
)

fig.show()

In [15]:
teams_per_year = df_editions[["year","teams"]]
fig = px.line(
    teams_per_year, 
    x="year",
    y="teams",
    title="Total Teams per Year",
    markers=True
)

fig.show()

In [16]:
fig = make_subplots(rows=1, cols=2)

fig1 = px.box(
    attendance_per_year,
    points="all",
    y="attendance",
    hover_data=["year", "edition"],
    title="Total Attendance per Year",
)

fig2 = px.box(
    goals_per_year,
    points="all",
    y="goals",
    hover_data=["year", "edition"],
    title="Total Goals per Year",
)

for trace in fig1.data:
    fig.add_trace(trace, row=1, col=1)

for trace in fig2.data:
    fig.add_trace(trace, row=1, col=2)

fig.update_layout(showlegend=False)

fig.show()
# no outlires!

In [17]:
df_editions[
    ["year", "teams", "matches", "goals", "goals_per_match"]
].sort_values("goals_per_match", ascending=False).head()

,year,teams,matches,goals,goals_per_match
4,1954,16,26,140,5.38
2,1938,15,18,84,4.67
1,1934,16,17,70,4.12
3,1950,13,22,88,4.00
0,1930,13,18,70,3.89


In [18]:
df_editions["decade"] = (
    df_editions["year"] // 10 * 10
)

decade_stats = (
    df_editions.groupby("decade")
    .agg(
        editions=("year", "count"),
        avg_goals_per_match=("goals_per_match", "mean"),
        total_goals=("goals", "sum"),
        total_matches=("matches", "sum")
    )
    .reset_index()
)

decade_stats

,decade,editions,avg_goals_per_match,total_goals,total_matches
0,1930,3,4.226667,224,53
1,1950,3,4.326667,354,83
2,1960,2,2.780000,178,64
3,1970,3,2.733333,294,108
4,1980,2,2.675000,278,104
5,1990,3,2.530000,427,168
6,2000,2,2.410000,308,128
7,2010,3,2.526667,485,192
8,2020,1,2.690000,172,64


In [19]:
decade_stats.sort_values(
    "avg_goals_per_match",
    ascending=False
)

,decade,editions,avg_goals_per_match,total_goals,total_matches
1,1950,3,4.326667,354,83
0,1930,3,4.226667,224,53
2,1960,2,2.780000,178,64
3,1970,3,2.733333,294,108
8,2020,1,2.690000,172,64
4,1980,2,2.675000,278,104
5,1990,3,2.530000,427,168
7,2010,3,2.526667,485,192
6,2000,2,2.410000,308,128


In [20]:
fig = px.line(
    df_editions.sort_values("year"),
    x="year",
    y="goals_per_match",
    markers=True,
    title="Goals per Match Across World Cup History",
    labels={
        "year": "World Cup Year",
        "goals_per_match": "Average Goals per Match"
    }
)

fig.show()

### part 3

In [21]:
titles = df_editions["champion_canon"].value_counts().reset_index()
titles.columns = ["team", "titles"]
px.bar(titles, x="team", y="titles", title="World Cup Titles by Team")

In [22]:
finals = pd.concat([df_editions["champion_canon"], df_editions["runner_up_canon"]], ignore_index=True)
finals = finals.value_counts().reset_index()
finals.columns = ["team", "Finals Reached"]
px.bar(finals, x="team", y="Finals Reached", title="Finals reached by Team")

In [23]:

appearance = (
    pd.concat([df_matches["team1"], df_matches["team2"]], ignore_index=True)
    .value_counts()
    .rename_axis("team")
    .reset_index(name="appearance")
)
fig = px.bar(appearance, x="appearance", y="team",orientation='h' , title="World Cup Appearances by Team")
fig.show()


In [27]:
goals = (
    pd.concat([
        df_matches.groupby("team1")["score1"].sum().rename("goals"),
        df_matches.groupby("team2")["score2"].sum().rename("goals")
    ])
    .groupby(level=0)
    .sum()
    .rename_axis("team")
    .reset_index()
    .sort_values("goals", ascending=False)
)
goals = goals[goals["goals"] > 0]  # Filter out teams with zero goals
fig = px.bar(
    goals,
    x="goals",
    y="team",
    orientation="h",
    title="World Cup Goals by Team"
)
fig.show()

In [28]:
goals_against = (
    pd.concat([
        df_matches.groupby("team1")["score2"].sum().rename("goals_against"),
        df_matches.groupby("team2")["score1"].sum().rename("goals_against")
    ])
    .groupby(level=0)
    .sum()
    .rename_axis("team")
    .reset_index()
    .sort_values("goals_against", ascending=False)
)
fig = px.bar(
    goals_against,
    x="goals_against",
    y="team",
    orientation="h",
    title="World Cup Goals by Opponent Team"
)
fig.show()

In [31]:
wins = (
    df_matches["winner"]
    .dropna()
    .value_counts()
    .rename_axis("team")
    .reset_index(name="wins")
)

fig = px.bar(
    wins,
    x="wins",
    y="team",
    orientation="h",
    title="World Cup Wins by Team"
)
fig.show()

### part 4

**Note: df_matches only has 184 out 964 matches that took place in the world cup, so information from upcoming charts are not reliable..**

In [33]:
df_matches["stage_type"].unique()

<StringArray>
['Group', 'Knockout', 'Second group']
Length: 3, dtype: str

In [39]:
goal_per_stage = df_matches.groupby("stage_type")["total_goals"].sum().reset_index(name="goals").sort_values("goals", ascending=False)
fig = px.pie(
    goal_per_stage,
    names="stage_type",
    values="goals",
    hole=0.3,
    title="Total goals per Stage"
)
fig.show()

In [40]:
goal_per_stage = df_matches.groupby("stage_clean")["total_goals"].sum().reset_index(name="goals").sort_values("goals", ascending=False)
fig = px.pie(
    goal_per_stage,
    names="stage_clean",
    values="goals",
    hole=0.3,
    title="Total goals per Stage"
)
fig.show()

In [50]:
goal_per_stage = df_matches["total_goals"].value_counts().reset_index(name="count").sort_values("total_goals", ascending=True)
fig = px.bar(
    x="total_goals",
    y="count",
    data_frame=goal_per_stage,
    title="Distribution of Total Goals per Match"
)
fig.show()

### part 5

In [59]:
champions = df_editions["champion_canon"].value_counts()
runner_ups = df_editions["runner_up_canon"].value_counts()
third_place_canon = df_editions["third_place_canon"].value_counts()

finals = champions.add(runner_ups, fill_value=0).astype(int).sort_values(ascending=False)
top3 = finals.add(third_place_canon, fill_value=0).astype(int).sort_values(ascending=False)

titles_df = champions.rename_axis("team").reset_index(name="count")
finals_df = finals.rename_axis("team").reset_index(name="count")
top3_df = top3.rename_axis("team").reset_index(name="count")

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["Titles", "Finals reached", "Top-3 finishes"],
    shared_yaxes=True,
)
for i, d in enumerate([titles_df, finals_df, top3_df]):
    fig.add_trace(px.bar(d, x="team", y="count").data[0], row=1, col=i + 1)

fig.update_layout(showlegend=False, title="World Cup Success: Titles vs Finals vs Top 3")
fig.show()

In [65]:
champions[champions > 2].cumsum() / champions.sum() * 100

champion_canon
Brazil       22.727273
Italy        40.909091
Germany      59.090909
Argentina    72.727273
Name: count, dtype: float64

In [66]:
big_teams = champions[champions >= 2].index

df_editions["big_on_podium"] = (
    df_editions["champion_canon"].isin(big_teams).astype(int)
    + df_editions["runner_up_canon"].isin(big_teams).astype(int)
    + df_editions["third_place_canon"].isin(big_teams).astype(int)
)

In [67]:
decade_dom = df_editions.groupby("decade").agg(
    editions=("year", "count"),
    big_slots=("big_on_podium", "sum"),
    distinct_champions=("champion_canon", "nunique"),
).reset_index()

decade_dom["big_share_pct"] = decade_dom["big_slots"] / (decade_dom["editions"] * 3) * 100
decade_dom

,decade,editions,big_slots,distinct_champions,big_share_pct
0,1930,3,6,2,66.666667
1,1950,3,5,3,55.555556
2,1960,2,2,2,33.333333
3,1970,3,6,3,66.666667
4,1980,2,5,2,83.333333
5,1990,3,7,3,77.777778
6,2000,2,5,2,83.333333
7,2010,3,4,3,44.444444
8,2020,1,2,1,66.666667


In [68]:
fig = px.bar(
    decade_dom, x="decade", y="big_share_pct",
    text="editions",
    title="Share of Podium Places Taken by Big Teams, per Decade",
    labels={"big_share_pct": "% of podium places", "decade": "Decade"},
)
fig.update_xaxes(type="category")   
fig.update_yaxes(range=[0, 100])
fig.show()

### part 6

In [5]:
fig = px.bar(
    data_frame=df_scorers,
    x="goals",
    y="year",
    orientation="h",
    title="Top Goal Scorers",
)
fig.show()

In [6]:
top_scorer_countries = (
    df_scorers.groupby("country_canon")
    .size()
    .rename_axis("country")
    .rename("top_scorers")
    .sort_values(ascending=False)
    .reset_index()
)

fig = px.bar(
    data_frame=top_scorer_countries,
    x="country",
    y="top_scorers",
    title="Top Countries that Produced Top scorers"
)

fig.show()


In [8]:
df_scorers["team_result"].unique()

<StringArray>
['Runner-up', 'Third place', nan, 'Champions', 'Quarter-final',
 'Fourth place']
Length: 6, dtype: str

In [9]:
rank_map = {"Champions": 1, "Runner-up": 2, "Third place": 3, "Fourth place": 4, "Quarter-final": 5, "Semi-final": 6}
df_scorers["result_rank"] = df_scorers["team_result"].map(rank_map)

In [10]:
valid = df_scorers.dropna(subset=["result_rank"]) 
pct_top3 = (valid["result_rank"] <= 3).mean() * 100
print(f"{pct_top3:.1f}% of Golden Boot winners were on a top-3 team")

81.0% of Golden Boot winners were on a top-3 team


In [11]:
fig = px.bar(valid["team_result"].value_counts().reset_index(), x="team_result", y="count")

fig.show()

In [15]:
cols = ["goals", "matches_played", "year"]
corr = df_scorers[cols].corr()

px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1, title="Correlation Heatmap")

### part 7

In [34]:
confederations = df_teams.groupby("confederation").agg(teams=("team", "count"))

px.pie(confederations, values="teams", names=confederations.index, title="Teams by Confederation", hole=0.3)

In [33]:
confederation_ranks = df_teams.groupby("confederation").agg(
    avg_rank=("fifa_rank", "mean")
)
px.bar(confederation_ranks, x=confederation_ranks.index, y="avg_rank", title="Average FIFA Rank by Confederation (Lowest is Best)", labels={"avg_rank": "Average FIFA Rank", "confederation": "Confederation"})

In [27]:
group_rank = df_teams.groupby("group").agg(
    avg_rank=("fifa_rank", "mean") # lower mean = better rank
).sort_values("avg_rank", ascending=True)
px.bar(group_rank, x=group_rank.index, y="avg_rank", title="Average FIFA Rank by Group (Lowest is Best)", labels={"avg_rank": "Average FIFA Rank", "group": "Group"})

In [41]:
group_rank_std = (
    df_teams.groupby("group")
    .agg(std_rank=("fifa_rank", "std"))
    .sort_values("std_rank")
)

px.scatter(
    group_rank_std,
    x=group_rank_std.index,
    y="std_rank",
    title="Standard Deviation of FIFA Rank by Group (Lower is More Balanced)",
    labels={
        "group": "Group",
        "std_rank": "Standard Deviation of FIFA Rank"
    }
)

In [48]:
match_expectations = df_fixtures[df_fixtures["stage"] == "Group Stage"]
match_expectations["expectation"] = abs(match_expectations["team1_fifa_rank"] - match_expectations["team2_fifa_rank"])
match_expectations["avg_teams_rank"] = (match_expectations["team1_fifa_rank"] + match_expectations["team2_fifa_rank"]) / 2


In [ ]:
match_expectations[["team1","team2","avg_teams_rank", "expectation"]].sort_values(["avg_teams_rank", "expectation"],ascending=True)
# top matches according to both 1- diff between ranks and 2- mean of said ranks

,team1,team2,avg_teams_rank,expectation
12,Brazil,Morocco,7.0,2.0
66,England,Croatia,7.5,7.0
48,France,Senegal,7.5,13.0
64,Colombia,Portugal,9.0,8.0
47,Uruguay,Spain,9.5,15.0
...,...,...,...,...
37,Iran,New Zealand,58.0,74.0
67,Ghana,Panama,59.0,12.0
39,New Zealand,Egypt,62.0,66.0
46,Cape Verde,Saudi Arabia,63.5,13.0


In [56]:
df_fixtures[df_fixtures["city"] == "San Francisco"]

,group,stage,team1,team2,venue,city,country,date,kickoff_et,team1_confederation,team1_fifa_rank,team1_coach,team2_confederation,team2_fifa_rank,team2_coach,manually_corrected
23,D,Group Stage,Paraguay,Australia,Levi's Stadium,San Francisco,USA,2026-06-25,22:00 ET,CONMEBOL,64.0,Gustavo Alfaro,AFC,26.0,Tony Popovic,True


In [65]:
venue_counts = (
    df_fixtures
    .groupby(["country", "city"])
    .size()
    .reset_index(name="matches")
)

assert venue_counts["matches"].sum() == len(df_fixtures) == 104

fig = px.sunburst(
    venue_counts,
    path=["country", "city"],   
    values="matches",           
    title="World Cup 2026: Matches by Host Country and City",
)

fig.update_traces(textinfo="label+percent parent")
fig.show()

### part 8